# Importing Libraries 

In [2]:
import os
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import joblib
 
from pathlib import Path
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_val_predict
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import accuracy_score, log_loss, brier_score_loss, confusion_matrix, classification_report

import warnings
warnings.filterwarnings("ignore")

In [3]:
 
base_dir        = Path('../artifacts')
PROCESSED_DIR   = base_dir / 'processed_data'
MODEL_DIR       = base_dir / 'models'
FIGURE_DIR      = base_dir / 'figures'
METRICS_DIR     = base_dir / 'metrics'
INTERIM_DIR     = base_dir / 'interim_data'
PREDICTIONS_DIR = base_dir / 'predictions'
 
for path in [MODEL_DIR, FIGURE_DIR, METRICS_DIR, INTERIM_DIR, PREDICTIONS_DIR]:
    path.mkdir(parents=True, exist_ok=True)

# Load Validation Predictions from All Three Models 

In [4]:
poisson_preds = pd.read_csv(f'{PREDICTIONS_DIR}/poisson_valid_preds.csv',
                             parse_dates=['date'])
dc_preds      = pd.read_csv(f'{PREDICTIONS_DIR}/dc_valid_preds.csv',
                             parse_dates=['date'])
xgb_preds     = pd.read_csv(f'{PREDICTIONS_DIR}/xgb_valid_preds.csv',
                             parse_dates=['date'])
 
print(f"Poisson preds : {poisson_preds.shape}")
print(f"DC preds      : {dc_preds.shape}")
print(f"XGBoost preds : {xgb_preds.shape}")
 
 
print("\nSample Poisson preds:")
print(poisson_preds[['date','home_team','away_team',
                      'prob_home_win','prob_draw','prob_away_win',
                      'result']].head(5).to_string(index=False))

Poisson preds : (1776, 34)
DC preds      : (1776, 12)
XGBoost preds : (1776, 12)

Sample Poisson preds:
      date   home_team     away_team  prob_home_win  prob_draw  prob_away_win  result
2024-01-12       Qatar       Lebanon       0.853465   0.109086       0.036929       0
2024-01-13       China    Tajikistan       0.485605   0.261476       0.252915       1
2024-01-13   Australia         India       0.767078   0.161622       0.071183       0
2024-01-13  Uzbekistan         Syria       0.465251   0.273701       0.261047       1
2024-01-13 Ivory Coast Guinea-Bissau       0.654733   0.223565       0.121685       0


# Build Meta-Feature Matrix 

In [5]:
meta = pd.DataFrame({
    # Poisson
    'p_hw':   poisson_preds['prob_home_win'],
    'p_d':    poisson_preds['prob_draw'],
    'p_aw':   poisson_preds['prob_away_win'],
    # Dixon-Coles
    'dc_hw':  dc_preds['prob_home_win'],
    'dc_d':   dc_preds['prob_draw'],
    'dc_aw':  dc_preds['prob_away_win'],
    # XGBoost
    'xgb_hw': xgb_preds['prob_home_win'],
    'xgb_d':  xgb_preds['prob_draw'],
    'xgb_aw': xgb_preds['prob_away_win'],
})
 
y_true = poisson_preds['result'].values   # 0=HomeWin, 1=Draw, 2=AwayWin
 
print(f"\nMeta-feature matrix shape : {meta.shape}")
print(meta.describe().round(4))


Meta-feature matrix shape : (1776, 9)
            p_hw        p_d       p_aw      dc_hw       dc_d      dc_aw  \
count  1776.0000  1776.0000  1776.0000  1776.0000  1776.0000  1776.0000   
mean      0.4751     0.2119     0.3081     0.4684     0.2305     0.3011   
std       0.2600     0.0792     0.2405     0.2648     0.0861     0.2444   
min       0.0000     0.0000     0.0000     0.0000     0.0000     0.0000   
25%       0.2758     0.1666     0.1110     0.2639     0.1812     0.1023   
50%       0.4632     0.2425     0.2565     0.4525     0.2639     0.2448   
75%       0.6816     0.2751     0.4452     0.6732     0.2993     0.4339   
max       0.9599     0.3048     0.9506     1.0000     0.3304     1.0000   

          xgb_hw      xgb_d     xgb_aw  
count  1776.0000  1776.0000  1776.0000  
mean      0.4669     0.2147     0.3176  
std       0.2570     0.0717     0.2369  
min       0.0020     0.0056     0.0010  
25%       0.2563     0.1680     0.1182  
50%       0.4616     0.2352     0.2641 

# Helper 

In [6]:
def predict_outcome(p_home, p_draw, p_away, draw_threshold=0.28):
    if p_draw >= draw_threshold:
        return 1
    return 0 if p_home >= p_away else 2
 
 
def tune_threshold(prob_home, prob_draw, prob_away, y_true):
    """Find best draw threshold on validation set."""
    best_thresh, best_acc = 0.28, 0.0
    for thresh in np.arange(0.20, 0.45, 0.01):
        preds = [predict_outcome(h, d, a, thresh)
                 for h, d, a in zip(prob_home, prob_draw, prob_away)]
        acc = accuracy_score(y_true, preds)
        if acc > best_acc:
            best_acc, best_thresh = acc, thresh
    return best_thresh, best_acc
 
 
def evaluate(prob_home, prob_draw, prob_away, y_true,
             draw_threshold=0.28, label='Model'):
    """Full evaluation: accuracy, log loss, Brier, confusion matrix."""
    preds = [predict_outcome(h, d, a, draw_threshold)
             for h, d, a in zip(prob_home, prob_draw, prob_away)]
 
    prob_matrix = np.column_stack([prob_home, prob_draw, prob_away])
 
    acc    = accuracy_score(y_true, preds)
    ll     = log_loss(y_true, prob_matrix, labels=[0, 1, 2])
    brier  = np.mean([
        brier_score_loss((y_true == c).astype(int), prob_matrix[:, c])
        for c in range(3)
    ])
 
    print(f"\n{'='*60}")
    print(f"{label}  (threshold={draw_threshold:.2f})")
    print(f"{'='*60}")
    print(f"  Accuracy   : {acc:.4f}  ({acc:.2%})")
    print(f"  Log Loss   : {ll:.4f}  (lower = better)")
    print(f"  Brier Score: {brier:.4f}  (lower = better)")
 
    cm = confusion_matrix(y_true, preds, labels=[0, 1, 2])
    print(pd.DataFrame(
        cm,
        index  =['Actual HW', 'Actual Draw', 'Actual AW'],
        columns=['Pred HW',   'Pred Draw',   'Pred AW']
    ))
    print(classification_report(
        y_true, preds,
        target_names=['HomeWin', 'Draw', 'AwayWin'],
        zero_division=0
    ))
 
    return {
        'label':         label,
        'threshold':     draw_threshold,
        'accuracy':      acc,
        'log_loss':      ll,
        'brier_score':   brier,
        'predictions':   preds,
        'prob_home_win': prob_home,
        'prob_draw':     prob_draw,
        'prob_away_win': prob_away,
    }
 

# Load Tuned Thresholds from Individual Models

In [7]:
with open(f'{METRICS_DIR}/poisson_threshold.json') as f:
    p_thresh  = json.load(f)['draw_threshold']
 
with open(f'{METRICS_DIR}/dc_threshold.json') as f:
    dc_thresh = json.load(f)['draw_threshold']
 
with open(f'{METRICS_DIR}/xgb_threshold.json') as f:
    xgb_thresh = json.load(f)['draw_threshold']
 
print(f"\nIndividual model thresholds:")
print(f"  Poisson  : {p_thresh}")
print(f"  DC       : {dc_thresh}")
print(f"  XGBoost  : {xgb_thresh}")


Individual model thresholds:
  Poisson  : 0.28
  DC       : 0.3
  XGBoost  : 0.31


# Evaluate Individual Models (Baseline) 

In [8]:
results = {}
 
results['Poisson'] = evaluate(
    poisson_preds['prob_home_win'],
    poisson_preds['prob_draw'],
    poisson_preds['prob_away_win'],
    y_true, draw_threshold=p_thresh, label='Poisson'
)
 
results['DC'] = evaluate(
    dc_preds['prob_home_win'],
    dc_preds['prob_draw'],
    dc_preds['prob_away_win'],
    y_true, draw_threshold=dc_thresh, label='Dixon-Coles'
)
 
results['XGBoost'] = evaluate(
    xgb_preds['prob_home_win'],
    xgb_preds['prob_draw'],
    xgb_preds['prob_away_win'],
    y_true, draw_threshold=xgb_thresh, label='XGBoost'
)


Poisson  (threshold=0.28)
  Accuracy   : 0.8102  (81.02%)
  Log Loss   : 0.6475  (lower = better)
  Brier Score: 0.1184  (lower = better)
             Pred HW  Pred Draw  Pred AW
Actual HW        769         55        1
Actual Draw      164        210       39
Actual AW         20         58      460
              precision    recall  f1-score   support

     HomeWin       0.81      0.93      0.87       825
        Draw       0.65      0.51      0.57       413
     AwayWin       0.92      0.86      0.89       538

    accuracy                           0.81      1776
   macro avg       0.79      0.77      0.77      1776
weighted avg       0.80      0.81      0.80      1776


Dixon-Coles  (threshold=0.30)
  Accuracy   : 0.8125  (81.25%)
  Log Loss   : 0.6320  (lower = better)
  Brier Score: 0.1167  (lower = better)
             Pred HW  Pred Draw  Pred AW
Actual HW        750         75        0
Actual Draw      127        264       22
Actual AW         15         94      429
         

# Stacking  Logistic Regression Meta-Model

Stacking meta-model: Logistic Regression trained on 9 probability outputs from Poisson + DC + XGBoost. Out-of-fold cross-validation prevents data leakage. Learns to weight each model's predictions per outcome class — typically trusting DC most for draws, Poisson/DC equally for HomeWin/AwayWin, XGBoost least.

In [9]:
X_meta = meta.values
scaler_meta = StandardScaler()
X_meta_scaled = scaler_meta.fit_transform(X_meta)
 
# Meta-model: Logistic Regression with L2 regularisation
meta_model = LogisticRegression(
    C=1.0,
    max_iter=1000,
    random_state=42,
    solver='lbfgs',             
)
 
# Get out-of-fold probability predictions on validation set
skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
meta_probs_oof = cross_val_predict(
    meta_model, X_meta_scaled, y_true,
    cv=skf, method='predict_proba'
)
 
# meta_probs_oof columns: [P(HomeWin), P(Draw), P(AwayWin)]
stk_hw = meta_probs_oof[:, 0]
stk_d  = meta_probs_oof[:, 1]
stk_aw = meta_probs_oof[:, 2]
 
# Tune threshold
best_thresh_stk, best_acc_stk = tune_threshold(stk_hw, stk_d, stk_aw, y_true)
 
results['Stacking'] = evaluate(
    stk_hw, stk_d, stk_aw,
    y_true, draw_threshold=best_thresh_stk,
    label='Stacking (Logistic Regression)'
)
 
# Now train final meta-model on ALL validation data for inference
meta_model.fit(X_meta_scaled, y_true)
 
print(f"\nMeta-model coefficients (9 features → 3 classes):")
coef_df = pd.DataFrame(
    meta_model.coef_,
    index  =['HomeWin', 'Draw', 'AwayWin'],
    columns=meta.columns
)
print(coef_df.round(4).to_string())


Stacking (Logistic Regression)  (threshold=0.40)
  Accuracy   : 0.9189  (91.89%)
  Log Loss   : 0.2136  (lower = better)
  Brier Score: 0.0402  (lower = better)
             Pred HW  Pred Draw  Pred AW
Actual HW        768         56        1
Actual Draw       17        378       18
Actual AW          1         51      486
              precision    recall  f1-score   support

     HomeWin       0.98      0.93      0.95       825
        Draw       0.78      0.92      0.84       413
     AwayWin       0.96      0.90      0.93       538

    accuracy                           0.92      1776
   macro avg       0.91      0.92      0.91      1776
weighted avg       0.93      0.92      0.92      1776


Meta-model coefficients (9 features → 3 classes):
           p_hw     p_d    p_aw   dc_hw    dc_d   dc_aw  xgb_hw   xgb_d  xgb_aw
HomeWin  2.5560 -0.2762 -2.6952  2.5478 -0.3672 -2.6326 -1.7288  0.2366  1.7763
Draw    -0.0996  0.4649 -0.0290 -0.1249  0.5259 -0.0484  0.0425 -0.0164 -0.0524
Aw

Stacking achieves 91.89% accuracy — a 10% improvement over best individual model. Draw recall jumps from 64% (DC) to 92% as the meta-model learns that  combined Poisson+DC draw signal is highly reliable.Coefficient analysis confirms: Poisson+DC trusted for all outcomes, XGBoost used mainly as a contrarian signal to detect when base models overfit.

# Saving the Ensemble Model 

In [10]:
# Save (Stacking only) 
joblib.dump(meta_model,  f'{MODEL_DIR}/ensemble_meta_model.pkl')
joblib.dump(scaler_meta, f'{MODEL_DIR}/ensemble_meta_scaler.pkl')

ensemble_cfg = {
    'final_method':        'stacking',
    'draw_threshold':       float(best_thresh_stk),
    'validation_accuracy':  float(results['Stacking']['accuracy']),
    'log_loss':             float(results['Stacking']['log_loss']),
    'brier_score':          float(results['Stacking']['brier_score']),
    'individual_accuracy': {
        'poisson':  float(results['Poisson']['accuracy']),
        'dc':       float(results['DC']['accuracy']),
        'xgboost':  float(results['XGBoost']['accuracy']),
        'stacking': float(results['Stacking']['accuracy']),
    }
}

with open(f'{METRICS_DIR}/ensemble_config.json', 'w') as f:
    json.dump(ensemble_cfg, f, indent=2)

ensemble_preds = poisson_preds[['date', 'home_team', 'away_team',
                                 'home_goals', 'away_goals', 'result']].copy()
ensemble_preds['prob_home_win'] = stk_hw
ensemble_preds['prob_draw']     = stk_d
ensemble_preds['prob_away_win'] = stk_aw
ensemble_preds['pred_result']   = [
    predict_outcome(h, d, a, best_thresh_stk)
    for h, d, a in zip(stk_hw, stk_d, stk_aw)
]
ensemble_preds.to_csv(f'{PREDICTIONS_DIR}/ensemble_valid_preds.csv', index=False)